In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline

In [18]:
pd.set_option('display.max_columns', None)

In [19]:
df = pd.read_csv('C:/Users/user/ML/project1/data/heart_disease.csv')
df.head()

,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,Low HDL Cholesterol,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Heart Disease Status
0,56.0,Male,153.0,155.0,High,Yes,Yes,No,24.991591,Yes,Yes,No,High,Medium,7.633228,Medium,342.0,NaN,12.969246,12.387250,No
1,69.0,Female,146.0,286.0,High,No,Yes,Yes,25.221799,No,Yes,No,Medium,High,8.744034,Medium,133.0,157.0,9.355389,19.298875,No
2,46.0,Male,126.0,216.0,Low,No,No,No,29.855447,No,Yes,Yes,Low,Low,4.440440,Low,393.0,92.0,12.709873,11.230926,No
3,32.0,Female,122.0,293.0,High,Yes,Yes,No,24.130477,Yes,No,Yes,Low,High,5.249405,High,293.0,94.0,12.509046,5.961958,No
4,60.0,Male,166.0,242.0,Low,Yes,Yes,Yes,20.486289,Yes,No,No,Low,High,7.030971,High,263.0,154.0,10.381259,8.153887,No


In [20]:
yes_no_columns = ['Smoking', 'Family Heart Disease', 'Diabetes', 
                    'High Blood Pressure', 'Low HDL Cholesterol',
                    'High LDL Cholesterol', 'Heart Disease Status']
df[yes_no_columns] = (df[yes_no_columns]=='Yes').astype(int)
df

,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,Low HDL Cholesterol,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Heart Disease Status
0,56.0,Male,153.0,155.0,High,1,1,0,24.991591,1,1,0,High,Medium,7.633228,Medium,342.0,NaN,12.969246,12.387250,0
1,69.0,Female,146.0,286.0,High,0,1,1,25.221799,0,1,0,Medium,High,8.744034,Medium,133.0,157.0,9.355389,19.298875,0
2,46.0,Male,126.0,216.0,Low,0,0,0,29.855447,0,1,1,Low,Low,4.440440,Low,393.0,92.0,12.709873,11.230926,0
3,32.0,Female,122.0,293.0,High,1,1,0,24.130477,1,0,1,Low,High,5.249405,High,293.0,94.0,12.509046,5.961958,0
4,60.0,Male,166.0,242.0,Low,1,1,1,20.486289,1,0,0,Low,High,7.030971,High,263.0,154.0,10.381259,8.153887,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,25.0,Female,136.0,243.0,Medium,1,0,0,18.788791,1,0,1,Medium,High,6.834954,Medium,343.0,133.0,3.588814,19.132004,1
9996,38.0,Male,172.0,154.0,Medium,0,0,0,31.856801,1,0,1,NaN,High,8.247784,Low,377.0,83.0,2.658267,9.715709,1
9997,73.0,Male,152.0,201.0,High,1,0,1,26.899911,0,1,1,NaN,Low,4.436762,Low,248.0,88.0,4.408867,9.492429,1
9998,23.0,Male,142.0,299.0,Low,1,0,1,34.964026,1,0,1,Medium,High,8.526329,Medium,113.0,153.0,7.215634,11.873486,1


In [21]:
X = df.drop('Heart Disease Status', axis=1)
y = df['Heart Disease Status']

In [22]:
y.value_counts()

Heart Disease Status
0    8000
1    2000
Name: count, dtype: int64

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, 
                                                    stratify=y)
X_train.shape[0], X_test.shape[0]

(7500, 2500)

**train**

In [24]:
X_train.isna().sum()

Age                       18
Gender                    10
Blood Pressure            14
Cholesterol Level         20
Exercise Habits           21
Smoking                    0
Family Heart Disease       0
Diabetes                   0
BMI                       19
High Blood Pressure        0
Low HDL Cholesterol        0
High LDL Cholesterol       0
Alcohol Consumption     1926
Stress Level              18
Sleep Hours               23
Sugar Consumption         24
Triglyceride Level        20
Fasting Blood Sugar       18
CRP Level                 18
Homocysteine Level        12
dtype: int64

In [25]:
num_columns = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
ordinal_columns = ['Exercise Habits', 'Stress Level', 'Sugar Consumption']
obj_columns = X_train.select_dtypes(include='object').columns.tolist()

cat_columns = [col for col in obj_columns if col not in ordinal_columns + ['Alcohol Consumption']]

In [26]:
ordinal_categories = {
    'Exercise Habits': ['Low', 'Medium', 'High'],
    'Stress Level': ['Low', 'Medium', 'High'],
    'Sugar Consumption': ['Low', 'Medium', 'High']
}
ordinal_categories_list = [ordinal_categories[col] for col in ordinal_columns]

In [27]:
num_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

ordinal_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(categories=ordinal_categories_list))
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]
)

alc_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]
)

In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_columns),
        ('ord', ordinal_pipeline, ordinal_columns),
        ('cat', cat_pipeline, cat_columns),
        ('alc', alc_pipeline, ['Alcohol Consumption'])
    ]
)


In [29]:
X_train_processed = preprocessor.fit_transform(X_train)
X_train_processed

array([[-0.51004405,  1.16191114, -0.70887984, ...,  1.        ,
         0.        ,  0.        ],
       [ 1.25095508, -0.4315774 , -0.01613029, ...,  0.        ,
         0.        ,  1.        ],
       [-0.01476304, -1.00068045, -1.28617113, ...,  1.        ,
         0.        ,  0.        ],
       ...,
       [ 1.03083019,  1.67410389,  0.23787788, ...,  0.        ,
         0.        ,  1.        ],
       [-1.66569973, -1.05759076, -1.14762122, ...,  0.        ,
         1.        ,  0.        ],
       [-1.06035628,  1.04809053,  1.36936882, ...,  1.        ,
         0.        ,  0.        ]])

In [30]:
feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_train_df.head()

,num__Age,num__Blood Pressure,num__Cholesterol Level,num__BMI,num__Sleep Hours,num__Triglyceride Level,num__Fasting Blood Sugar,num__CRP Level,num__Homocysteine Level,ord__Exercise Habits,ord__Stress Level,ord__Sugar Consumption,cat__Gender_Female,cat__Gender_Male,alc__Alcohol Consumption_High,alc__Alcohol Consumption_Low,alc__Alcohol Consumption_Medium
0,-0.510044,1.161911,-0.708880,0.038635,0.930033,0.417252,-0.352115,0.178106,-0.151517,2.0,2.0,2.0,0.0,1.0,1.0,0.0,0.0
1,1.250955,-0.431577,-0.016130,0.322679,1.489129,-0.228204,-0.904304,-0.431642,-0.744281,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
2,-0.014763,-1.000680,-1.286171,-1.430956,-0.992088,1.454591,-0.267163,-1.108867,-1.711276,0.0,2.0,2.0,0.0,1.0,1.0,0.0,0.0
3,-0.675138,1.332642,0.168603,-1.302980,1.254003,-1.300121,-0.139735,-1.635832,-0.837337,1.0,2.0,1.0,0.0,1.0,1.0,0.0,0.0
4,1.250955,1.560283,0.030053,-0.402728,-1.580333,-0.792977,-0.352115,-1.647085,-1.095865,0.0,0.0,2.0,0.0,1.0,1.0,0.0,0.0


**test**

In [31]:
X_test_processed = preprocessor.transform(X_test)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)
X_test_df.head()

,num__Age,num__Blood Pressure,num__Cholesterol Level,num__BMI,num__Sleep Hours,num__Triglyceride Level,num__Fasting Blood Sugar,num__CRP Level,num__Homocysteine Level,ord__Exercise Habits,ord__Stress Level,ord__Sugar Consumption,cat__Gender_Female,cat__Gender_Male,alc__Alcohol Consumption_High,alc__Alcohol Consumption_Low,alc__Alcohol Consumption_Medium
0,0.810705,-1.114501,0.676619,-0.050529,0.797350,0.544037,-1.414016,1.029258,1.365841,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
1,0.260393,1.161911,-0.108497,-1.027590,-1.536031,-0.401094,0.327502,-1.232203,-1.059670,2.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,0.425487,-0.886860,-1.078346,-0.502769,-1.080718,1.154915,-0.139735,-1.011980,1.726449,2.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0
3,-0.289919,0.877360,-0.431780,-1.206339,-0.073564,0.682349,-1.329064,0.032179,0.980237,2.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,1.030830,-0.147026,0.145511,0.253206,-0.592630,-1.657427,-1.201636,0.087549,-0.714843,2.0,0.0,2.0,1.0,0.0,0.0,0.0,1.0


In [32]:
pd.concat([X_train_df.reset_index(drop=True), 
           y_train.reset_index(drop=True)], axis=1).to_csv(
    'C:/Users/user/ML/project1/data/data_train.csv',
    index=False)

pd.concat([X_test_df.reset_index(drop=True),
            y_test.reset_index(drop=True)], axis=1).to_csv(
    'C:/Users/user/ML/project1/data/data_test.csv',
    index=False)